# Demo 2 — Orthogonal ensemble + intersections

Investor narrative: A simple but powerful fix: consensus intersection collapses polysemanticity while preserving task accuracy.


In [ ]:
import os, sys, json, random
import numpy as np
import torch, transformers, yaml, matplotlib
import matplotlib.pyplot as plt
import sklearn
print('torch', torch.__version__)
print('transformers', transformers.__version__)
print('numpy', np.__version__)
print('matplotlib', matplotlib.__version__)
print('yaml', yaml.__version__)
print('sklearn', sklearn.__version__)
print('Note: build py_nsi locally: cd py_nsi && maturin develop --release')


In [ ]:
sys.path.append(os.path.abspath('.'))
from python.utils.config import load_yaml
cfg = load_yaml('configs/demo2_ensemble.yaml')
cfg


In [ ]:
from python.datasets.bank_sentences import generate_bank_dataset
n_per_class = int(cfg['dataset']['n_per_class'])
seed = int(cfg['dataset']['seed'])
texts, labels = generate_bank_dataset(n_per_class=n_per_class, seed=seed)
idx0 = [i for i,l in enumerate(labels) if l==0][:3]
idx1 = [i for i,l in enumerate(labels) if l==1][:3]
for i in idx0: print('C0:', texts[i])
for i in idx1: print('C1:', texts[i])
num_concepts = len(set(labels))
labels_np = np.asarray(labels, dtype=np.int32)


In [ ]:
from python.activations.extract import get_model_and_tokenizer, capture_layer_activations
model_name = cfg['model_name']
layer_index = int(cfg['layer_index'])
model, tokenizer = get_model_and_tokenizer(model_name)
acts = capture_layer_activations(model, tokenizer, texts, layer_index=layer_index)
acts.shape


In [ ]:
from python.models.sae import train_sae, encode_topk
s_single = cfg['sae_single']
sae_model, train_stats = train_sae(
    acts,
    hidden_dim=int(s_single['hidden_dim']),
    top_k=int(s_single['top_k']),
    l1_lambda=float(s_single['l1_lambda']),
    seed=int(s_single['seed']),
    epochs=int(s_single['epochs']),
    lr=float(s_single['lr']),
    device='cpu',
)
plt.figure(figsize=(6,3));
plt.plot(train_stats['total_curve'], label='total');
plt.plot(train_stats['recon_curve'], label='recon');
plt.plot(train_stats['l1_curve'], label='l1');
plt.title('SAE training loss'); plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
from python.metrics.polysemanticity import concept_probs, poly_count, entropy, summarize_polysemanticity
active_thr = float(s_single['active_threshold'])
features_single = encode_topk(sae_model, acts, top_k=int(s_single['top_k']), device='cpu')
prob_single = concept_probs(features_single, labels_np, num_concepts=num_concepts, active_threshold=active_thr)
eps = float(cfg['metrics']['eps'])
poly_single = poly_count(prob_single, eps)
ent_single = entropy(prob_single)
summary_single = summarize_polysemanticity(prob_single, eps)
print('single summary:', summary_single)
plt.figure(figsize=(6,3)); plt.hist(poly_single, bins=int(cfg['metrics']['hist_bins']), color='#4C78A8', edgecolor='white'); plt.title('Single SAE poly(f)'); plt.tight_layout(); plt.show()


In [ ]:
from python.ensemble.intersection import build_pyensemble, encode_all_and_intersect
ens = cfg['ensemble']
os.environ['PY_NSI_INPUT_DIM'] = str(acts.shape[1])
try:
    ensemble = build_pyensemble(feature_dim=int(ens['feature_dim']), top_k=int(ens['top_k']), seeds=[int(s) for s in ens['seeds']])
except Exception as e:
    raise RuntimeError('py_nsi not available. Build it with: cd py_nsi && maturin develop --release') from e
masks_bool = encode_all_and_intersect(ensemble, acts=acts, threshold=float(ens['intersect_threshold']))
features_intersection = masks_bool.astype(np.float32)
prob_intersection = concept_probs(features_intersection, labels_np, num_concepts=num_concepts, active_threshold=0.5)
poly_intersection = poly_count(prob_intersection, eps)
plt.figure(figsize=(6,3));
plt.hist(poly_single, bins=int(cfg['metrics']['hist_bins']), alpha=0.7, label='Single');
plt.hist(poly_intersection, bins=int(cfg['metrics']['hist_bins']), alpha=0.6, label='Ensemble ∩');
plt.legend(); plt.title('Polysemanticity: Single vs. Intersection'); plt.tight_layout(); plt.show()


In [ ]:
from python.metrics.downstream import evaluate_logreg
acc_single = evaluate_logreg(features_single, labels_np, seed=int(cfg['dataset']['seed']))['accuracy']
acc_intersection = evaluate_logreg(features_intersection, labels_np, seed=int(cfg['dataset']['seed']))['accuracy']
table = {'single': {'accuracy': acc_single}, 'intersection': {'accuracy': acc_intersection}}
print(json.dumps(table, indent=2))


In [ ]:
from python.utils.artifacts import create_run_dir, dump_json, dump_yaml
from python.plots.hist import plot_histogram
from python.plots.compare import plot_dual_hist
run_dir = create_run_dir(base_dir=cfg['outputs']['base_dir'], run_tag=cfg['outputs'].get('run_tag'))
np.save(os.path.join(run_dir, 'probs_single.npy'), prob_single)
np.save(os.path.join(run_dir, 'poly_counts_single.npy'), poly_single)
np.save(os.path.join(run_dir, 'entropy_single.npy'), ent_single)
np.save(os.path.join(run_dir, 'probs_intersection.npy'), prob_intersection)
np.save(os.path.join(run_dir, 'poly_counts_intersection.npy'), poly_intersection)
np.save(os.path.join(run_dir, 'entropy_intersection.npy'), entropy(prob_intersection))
dump_yaml(cfg, os.path.join(run_dir, 'config.yaml'))
plot_histogram(poly_single, bins=int(cfg['metrics']['hist_bins']), title='Single SAE polysemanticity', path=os.path.join(run_dir, 'poly_hist_single.png'))
plot_histogram(poly_intersection, bins=int(cfg['metrics']['hist_bins']), title='Ensemble ∩ polysemanticity', path=os.path.join(run_dir, 'poly_hist_intersection.png'))
plot_dual_hist(poly_single, poly_intersection, bins=int(cfg['metrics']['hist_bins']), labels=('Single','Ensemble ∩'), title='Single vs. Intersection', path=os.path.join(run_dir, 'poly_hist_dual.png'))
compare = {
  'single': {'median_poly': float(np.median(poly_single)), 'p90_poly': float(np.percentile(poly_single,90)), 'monosemantic_rate': float((poly_single==1).sum()/len(poly_single)), 'accuracy': float(acc_single), 'active_threshold': float(cfg['sae_single']['active_threshold']), 'eps': float(cfg['metrics']['eps'])},
  'intersection': {'median_poly': float(np.median(poly_intersection)), 'p90_poly': float(np.percentile(poly_intersection,90)), 'monosemantic_rate': float((poly_intersection==1).sum()/len(poly_intersection)), 'accuracy': float(acc_intersection), 'active_threshold': 0.5, 'intersect_threshold': float(cfg['ensemble']['intersect_threshold']), 'eps': float(cfg['metrics']['eps'])}
}
dump_json(compare, os.path.join(run_dir, 'compare.json'))
print('Artifacts saved to', run_dir)


## Acceptance mapping
- Intersections computed via Rust core (py_nsi) and compared to single SAE.
- Metrics follow METRICS.md definitions: poly(f), monosemantic rate, entropy.
- Notebook generates histograms and saves arrays and JSONs into outputs/ensemble.
